In [ ]:
import pandas as pd

# ==============================
# FILE PATHS
# ==============================
bom_file = r"D:/Tushar/main_with_subs_only.xlsx"
indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

MONTH_COL = "Feb'26 QTY"   # confirmed from your file
DAYS_IN_MONTH = 28         # Feb 2026

print("Starting calculation for Feb'26...\n")

# ==============================
# READ FILES
# ==============================
bom_df = pd.read_excel(bom_file)
indent_df = pd.read_excel(indent_file)

bom_df.columns = bom_df.columns.str.strip()
indent_df.columns = indent_df.columns.str.strip()

# ==============================
# NORMALIZE CODES
# ==============================
def normalize(series):
    return series.astype(str).str.strip().str.upper()

indent_df['Part number'] = normalize(indent_df['Part number'])
bom_df['Sub_Label'] = normalize(bom_df['Sub_Label'])
bom_df['Main_Label'] = normalize(bom_df['Main_Label'])

# ==============================
# PREPARE INDENT LOOKUP
# ==============================
indent_df = indent_df[['Part number', MONTH_COL]].copy()
indent_df[MONTH_COL] = pd.to_numeric(indent_df[MONTH_COL], errors='coerce').fillna(0)

indent_df['Daily_Demand'] = indent_df[MONTH_COL] / DAYS_IN_MONTH

daily_lookup = dict(zip(indent_df['Part number'], indent_df['Daily_Demand']))

print(f"Loaded {len(daily_lookup)} switches\n")

# ==============================
# PREPARE BOM
# ==============================
bom_df = bom_df[['Main_Label', 'Sub_Label', 'Sub_Count']].copy()
bom_df['Sub_Count'] = pd.to_numeric(bom_df['Sub_Count'], errors='coerce').fillna(0)

# ==============================
# ROW-WISE ACCUMULATION
# ==============================
daily_totals = {}

for idx, row in bom_df.iterrows():

    child = row['Main_Label']
    switch = row['Sub_Label']
    usage = row['Sub_Count']

    daily_switch = daily_lookup.get(switch, 0)
    contribution = daily_switch * usage

    if child in daily_totals:
        daily_totals[child] += contribution
    else:
        daily_totals[child] = contribution

# ==============================
# FINAL RESULT
# ==============================
result = pd.DataFrame({
    'Child_Part': list(daily_totals.keys()),
    'Daily_Requirement': list(daily_totals.values())
})

result['Two_Day_Requirement'] = result['Daily_Requirement'] * 2
result = result.sort_values('Two_Day_Requirement', ascending=False).round(2)

print("\nTop child parts:")
print(result.head(15))

# ==============================
# SAVE
# ==============================
output_file = "Feb26_2Day_Child_Requirement.xlsx"
result.to_excel(output_file, index=False)

print(f"\nSaved to {output_file}")
